# LC 127 — Word Ladder

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Model each word as a graph node
with edges to all words differing by exactly one letter. The
shortest path from `beginWord` to `endWord` in this unweighted
graph is found by BFS — it guarantees minimum steps.
</div>

## Official Problem Statement

Given `beginWord`, `endWord`, and a `wordList`, return the
**number of words** in the shortest transformation sequence
from `beginWord` to `endWord`, where each step changes exactly
one letter and each intermediate word must be in `wordList`.
Return 0 if no path exists.

**Example 1:**
```
Input:  beginWord="hit", endWord="cog"
        wordList=["hot","dot","dog","lot","log","cog"]
Output: 5
Explanation: hit→hot→dot→dog→cog
```
**Example 2:**
```
Input:  beginWord="hit", endWord="cog"
        wordList=["hot","dot","dog","lot","log"]
Output: 0  (cog not in wordList)
```
**Constraints:**
- `1 <= beginWord.length <= 10`
- `endWord` and all words in `wordList` have the same length.
- `1 <= wordList.length <= 5000`

## What This Is Actually Asking

Each word is a node. Two words share an edge if they differ by
exactly one character. We want the shortest path (minimum edges)
from `beginWord` to `endWord`.

The naive approach generates all pairs and checks differences —
O(n² * L) for n words of length L. Better: for each word in the
queue, try replacing each position with each letter a-z and check
if the result is in the word set.

BFS naturally gives the shortest path in an unweighted graph.
We return the step count (number of words, not edges) when we
first reach `endWord`.

## Walk Through an Example by Hand

```
beginWord="hit", endWord="cog"
wordSet={"hot","dot","dog","lot","log","cog"}

BFS:
  queue=[("hit",1)]

  Step 1: word="hit", steps=1
    Try h_t: aat..zit, hat, hbt... → "hot" ∈ wordSet!
    → remove "hot", queue=["hot"] steps=2

  Step 2: word="hot", steps=2
    Try _ot: "dot" ∈ wordSet, "lot" ∈ wordSet
    → queue=["dot","lot"] steps=3

  Step 3: word="dot", steps=3
    Try d_t: nothing. Try do_: "dog" ∈ wordSet
    → queue=["lot","dog"] steps=4
  
  Step 3: word="lot", steps=3
    Try l_t: nothing. Try lo_: "log" ∈ wordSet
    → queue=["dog","log"] steps=4

  Step 4: word="dog", steps=4
    Try _og: "cog" ∈ wordSet!
    → return steps+1 = 5 ✓
```

## The Picture

```
Graph of one-letter transformations:

  hit
   |
  hot
  / \
dot   lot
 |     |
dog   log
  \   /
   cog   ← endWord

BFS frontier expands level by level:

  Level 1: {hit}        steps=1
  Level 2: {hot}        steps=2
  Level 3: {dot,lot}    steps=3
  Level 4: {dog,log}    steps=4
  Level 5: {cog}        steps=5 → FOUND

For each word, generate neighbors:
  word = "dot"
  pos 0: aat,bot,cot,...,zot  (check each)
  pos 1: dat,dbt,...,dzt
  pos 2: doa,dob,...,doz
  → O(L * 26) per word
```

## When To Use This Pattern

- When you need the **shortest path** in an unweighted graph,
  think **BFS** (not DFS).
- When the graph is implicit (nodes are generated on the fly),
  think **BFS with a visited set** to avoid re-exploring.
- When edges represent "one mutation away", think **generate
  neighbors by enumeration** (a-z substitution here).
- When the state space is large but most paths are irrelevant,
  think **remove from visited set as you enqueue** to avoid
  re-queueing.
- When the graph is sparse or edge generation is expensive,
  think **bidirectional BFS** to reduce search space.

## The Approach

Convert `wordList` to a set for O(1) lookup. Initialize a BFS
queue with `(beginWord, 1)`. For each word dequeued, try changing
each character position to each letter a-z. If the result equals
`endWord`, return `steps + 1`. If it's in `wordSet`, remove it
(to mark visited) and add `(newWord, steps+1)` to the queue.

Removing from the set instead of a separate visited set is a
clean optimization: each word is only processed once, and removal
prevents cycles.

In [ ]:
# Imports
from collections import deque
from typing import List

In [ ]:
# ----------------------------------------------------------
# Harness
# ----------------------------------------------------------
def test_harness(func):
    cases = [
        (
            "hit", "cog",
            ["hot","dot","dog","lot","log","cog"],
            5
        ),
        (
            "hit", "cog",
            ["hot","dot","dog","lot","log"],
            0
        ),
        (
            "a", "c",
            ["a","b","c"],
            2
        ),
        (
            "hot", "dog",
            ["hot","dog"],
            0
        ),
    ]
    passed = 0
    for begin, end, wl, expected in cases:
        result = func(begin, end, wl)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if not ok:
            print(f"{status} | {begin}→{end}"
                  f" | expected={expected}"
                  f" | got={result}")
        else:
            print(f"{status} | {begin}→{end}"
                  f" = {result} steps")
        passed += ok
    print(f"\n{passed}/{len(cases)} tests passed")

In [ ]:
def ladderLength(
    beginWord: str, endWord: str, wordList: List[str]
) -> int:
    """
    LC 127 — Word Ladder

    BFS on implicit word graph:
    - wordSet = set(wordList) for O(1) lookup
    - Queue: (word, steps), start with (beginWord, 1)
    - For each word: try changing each char to a-z
      - If newWord == endWord: return steps+1
      - If newWord in wordSet: remove, enqueue
    - Return 0 if queue exhausted

    Time:  O(M^2 * N) M=word length, N=wordList size
    Space: O(M^2 * N)
    """
    pass
    # Debug hints:
    # print(f"word={word} steps={steps}"
    #       f" trying={new_word}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(ladderLength)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| DFS (incorrect for shortest) | O(N!) | O(N) | Not guaranteed shortest |
| BFS (standard) | O(M²·N) | O(M·N) | M=len, N=wordlist |
| Bidirectional BFS | O(M²·N/2) | O(M·N) | ~2x faster in practice |
| A* with heuristic | O(M²·N) | O(M·N) | Complex, rarely needed |

## Real World Connection

**AWS / DE context:** Word Ladder is the template for any
"minimum edit path" problem in production systems. DNA sequence
alignment (one-base mutations), spell correction (edit distance
= 1), and routing table lookups all reduce to this BFS pattern.

At Citi, graph-based BFS underlies relationship discovery in
know-your-customer (KYC) networks: find the shortest chain of
ownership linking a suspicious entity to a known bad actor.

In AWS Neptune or graph databases, this is literally a shortest-
path query. Understanding the BFS mechanics helps you reason
about query plans and index strategies for graph workloads.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra